# Data Analysis and PreProcessing

## Setting up Environment

In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYSPARK_PYTHON'] = '/home/subha/miniconda3/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/home/subha/miniconda3/bin/python'

## Starting Pyspark with Master 10G Memory

In [2]:
import findspark
findspark.init('/home/subha/aiwork/spark')
# Initializing the spark context
#import pyspark.pandas as ps
#pdf_incidents = df_incidents.to_pandas_on_spark()
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col, lower, regexp_replace
from pyspark.sql.types import StringType, ArrayType
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from pyspark.sql.functions import *

# Configure Spark to use multiple threads
spark = SparkSession.builder.appName("Amazon Reviews Analysis")\
    .master("local[*]")\
    .config("spark.executorEnv.PYSPARK_PYTHON", "/home/subha/miniconda3/bin/python")\
    .config("spark.driver.maxResultSize","10g")\
    .config("spark.executor.instances", "4")\
    .config("spark.executor.cores", "2")\
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

25/03/23 10:36:34 WARN Utils: Your hostname, neoshiva resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/03/23 10:36:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/23 10:36:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Gathering Train and Test Data

### Reading Test and Train Datasets

In [3]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .csv("/home/subha/aiwork/project/amazon_train_dataset/Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv")
    
reviews = df.select( col("`reviews.rating`").alias("rating"), col("`reviews.text`").alias("review_text"))
df_sentiment_reviews_ = reviews.withColumn(
        "sentiment",
        when(col("rating") > 3, 1)
        .otherwise(0)
    )
df_sentiment_reviews = df_sentiment_reviews_.select("review_text","sentiment")
df_sentiment_reviews.show(5)

+--------------------+---------+
|         review_text|sentiment|
+--------------------+---------+
|I order 3 of them...|        0|
|Bulk is always th...|        1|
|Well they are not...|        1|
|Seem to work as w...|        1|
|These batteries a...|        1|
+--------------------+---------+
only showing top 5 rows



### Shaping the Datasets (Feature Split)

### Display and Counts

In [20]:
display(df_sentiment_reviews.show(5))

+--------------------+---------+
|         review_text|sentiment|
+--------------------+---------+
|I order 3 of them...|        0|
|Bulk is always th...|        1|
|Well they are not...|        1|
|Seem to work as w...|        1|
|These batteries a...|        1|
+--------------------+---------+
only showing top 5 rows



None

In [4]:
df_sentiment_reviews.count()

28332

## Natural Language Preprocessing

### NLTK resources

In [5]:
# Download required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/subha/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/subha/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/subha/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/subha/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

### Lemmetizer and Stop Words

In [10]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

### Clean, Tokenize and UDF creation

In [11]:
def clean_text(text):
    """
    Clean text by removing special characters, numbers, and converting to lowercase
    """
    if not text:
        return text
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def tokenize_and_preprocess(text):
    """
    Tokenize, remove stopwords, and lemmatize text
    """
    if not text:
        return []
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and lemmatize
    # tokens = [lemmatizer.lemmatize(token) for token in tokens 
    #          if token.lower() not in stop_words and len(token) > 2]
    tokens = [token for token in tokens 
             if token.lower() not in stop_words and len(token) > 2]
    
    return tokens
    
# Register UDFs
clean_text_udf = udf(clean_text, StringType())
tokenize_and_preprocess_udf = udf(tokenize_and_preprocess, ArrayType(StringType()))

### NLTK Review Process Method

In [12]:
def process_reviews_with_nltk(reviews_df):
    """
    Process reviews using NLTK for text cleaning and preprocessing
    
    Args:
        reviews_df: DataFrame with 'sentiment' and 'reviews' columns
    Returns:
        DataFrame with processed text
    """
    
    # Apply text cleaning
    processed_df = reviews_df.withColumn(
        "cleaned_text",
        clean_text_udf(col("review_text"))
    )
    
    # Apply tokenization, stopword removal, and lemmatization
    processed_df = processed_df.withColumn(
        "processed_tokens",
        tokenize_and_preprocess_udf(col("cleaned_text"))
    )
    
    # Convert tokens back to text
    processed_df = processed_df.withColumn(
        "processed_text",
        udf(lambda x: ' '.join(x) if x else '', StringType())(col("processed_tokens"))
    )
    
    return processed_df

## Executing the Preprocessing Steps

### Test and Train Spark DF created

In [13]:
test_train_processed_reviews_df = process_reviews_with_nltk(df_sentiment_reviews)

In [14]:
test_train_processed_reviews_df.show()

+--------------------+---------+--------------------+--------------------+--------------------+
|         review_text|sentiment|        cleaned_text|    processed_tokens|      processed_text|
+--------------------+---------+--------------------+--------------------+--------------------+
|I order 3 of them...|        0|i order of them a...|[order, one, item...|order one item ba...|
|Bulk is always th...|        1|bulk is always th...|[bulk, always, le...|bulk always less ...|
|Well they are not...|        1|well they are not...|[well, duracell, ...|well duracell pri...|
|Seem to work as w...|        1|seem to work as w...|[seem, work, well...|seem work well na...|
|These batteries a...|        1|these batteries a...|[batteries, long,...|batteries long la...|
|Bought a lot of b...|        1|bought a lot of b...|[bought, lot, bat...|bought lot batter...|
|ive not had any p...|        1|ive not had any p...|[ive, problame, b...|ive problame batt...|
|Well if you are l...|        1|well if 

In [15]:
test_train_processed_reviews_df.printSchema()

root
 |-- review_text: string (nullable = true)
 |-- sentiment: integer (nullable = false)
 |-- cleaned_text: string (nullable = true)
 |-- processed_tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- processed_text: string (nullable = true)



In [16]:
test_train_processed_reviews_df.count()

28332

### Writing to Output Location

In [38]:
test_processed_reviews_df.select("cleaned_text","sentiment").coalesce(1).write.mode("overwrite").parquet("output/cleandata/small/test_data")

In [39]:
train_processed_reviews_df.select("cleaned_text","sentiment").coalesce(1).write.mode("overwrite").parquet("output/cleandata/small/train_data")

### Creating Samples for hyper Parameter Tuning

#### Train Data

In [17]:
test_train_processed_reviews_df.createOrReplaceTempView("test_train_processed_reviews_df")

In [18]:
sql = """
select * 
from test_train_processed_reviews_df 
where sentiment = 0
"""
tr_te_df_0 = spark.sql(sql)
print(tr_te_df_0.count())
train_df_0, test_df_0 = tr_te_df_0.randomSplit([0.8, 0.2]) 

sql1 ="""
select * 
from test_train_processed_reviews_df 
where sentiment = 1  
"""
tr_te_df_1 = spark.sql(sql1)
print(tr_te_df_1.count())
train_df_1, test_df_1 = tr_te_df_1.randomSplit([0.8, 0.2]) 

u_train_df = train_df_0.union(train_df_1)
u_test_df = test_df_0.union(test_df_1)

2878
25454


In [19]:
print(train_df_0.count(),train_df_1.count(),test_df_0.count(),test_df_1.count())

2314 20406 564 5048


In [20]:
u_train_df.select("cleaned_text","sentiment").coalesce(1).write.mode("overwrite").parquet("output/cleandata/small/train_data_sample")
u_test_df.select("cleaned_text","sentiment").coalesce(1).write.mode("overwrite").parquet("output/cleandata/small/test_data_sample")
u_test_df.select("cleaned_text","sentiment").coalesce(1).write.mode("overwrite").parquet("output/cleandata/small/val_data_sample")

## Appendix

In [21]:
spark.stop()